In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/processed/cleaned_baywheels_df.csv")
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,start_hour,start_day,start_month,route_id
0,28929E5AFD15A2FB,electric_bike,2024-12-02 17:51:35.798,2024-12-02 17:54:29.841,O'Farrell St at Masonic Ave,SF-H16,Page St at Masonic Ave,SF-K16,37.781131,-122.447374,37.771135,-122.445341,member,17,Monday,12,SF-H16-SF-K16
1,5FC879CAC379C364,electric_bike,2024-12-30 12:01:54.081,2024-12-30 12:34:35.126,Greenwich St at Franklin St,SF-B21,Mason St at Halleck St,SF-A15,37.800287,-122.425786,37.803968,-122.455079,casual,12,Monday,12,SF-B21-SF-A15
2,7BF609D17E4AE925,electric_bike,2024-12-24 07:59:08.908,2024-12-24 08:01:14.056,San Francisco Public Library,SF-I24,Grove St at Gough St,SF-J22-2,37.778799,-122.415963,37.777870,-122.422953,member,7,Tuesday,12,SF-I24-SF-J22-2
3,9DEA637C58FBDDBA,electric_bike,2024-12-29 15:19:35.704,2024-12-29 15:26:32.882,Market St at Franklin St,SF-K22-1,Church St at Duboce Ave,SF-L20,37.773793,-122.421239,37.769818,-122.429148,member,15,Sunday,12,SF-K22-1-SF-L20
4,BD56E9D0CDA65000,classic_bike,2024-12-21 16:44:18.909,2024-12-21 16:46:07.511,Parker St at Fulton St,BK-F8,Parker St at Fulton St,BK-F8,37.862464,-122.264791,37.862464,-122.264791,member,16,Saturday,12,BK-F8-BK-F8


**Insights:**
1. Commuter Hubs are the high-volume zones. These station have the highest average and medium rides (15K-16K). This suggests that location is prime and everyone uses it. There's no big difference between classic and electric bike usage.
2. Hybrid Zones are the only zones with the massive jump between classic and electric bikes (avg: from 7600 to 8900 for members, med: from 5900 to 7900). Users are choosing electric bikes over classic ones more often.
3. Leisure zones have high volume with the highest Electric bike and Member average: ~17K. Even in leisure areas the combination of Member + Electric bike is the "gold standard" for usage.

**Conclusions:**
1. Commuter Hubs are the most stable. Need to be sure there are enough empty docks for arrivals and charged bikes for departure.
2. For increased revenue in Hybrid Zones older classic bikes might be replaced with electric ones.
3. Ensure the freshest, newest e-bikes are staged in Leisure Zones, especially on Friday and the weekend. Power users (Members) are using them heavily for short weekend trips and sighseeing.

### Growth Analysis
Classifying stations and their usage patterns for potential growth.

I classify stations in 3 groups: Commuters, Social/Leisure, Hybrid.
- Commuter Hub: High concentration of rides during rush hours (7AM-10AM/4PM-7PM)
- Leisure Zone: High concentration of rides on Weekends or late-night Weekdays (8PM-11PM)
- Hybrid: Balanced activity across the week

In [13]:
df["is_weekend"] = df["start_day"].isin(["Saturday", "Sunday"])
df["is_rush_hour"] = df["start_hour"].isin([7, 8, 9, 16, 17, 18])

station_profiles = df.groupby("start_station_name").agg(
    total_rides=("ride_id", "count"),
    commute_pct=("is_rush_hour", "mean"), # bool mean -> percent
    weekend_pct=("is_weekend", "mean")
).reset_index()

columns_to_pct = ["commute_pct", "weekend_pct"]
station_profiles[columns_to_pct] = station_profiles[columns_to_pct] * 100

station_profiles

,start_station_name,total_rides,commute_pct,weekend_pct
0,Sloat Blvd at The Great Highway to 46th Ave,262,28.244275,46.183206
1,10th Ave at E 15th St,474,34.177215,18.987342
2,10th Ave at Irving St,10681,39.996255,36.429173
3,10th St at Chestnut St,873,35.395189,21.420389
4,10th St at Empire St,555,43.063063,12.072072
...,...,...,...,...
637,Willow Park,692,59.826590,25.144509
638,Willow St at Blewett Ave,478,44.560669,37.029289
639,Willow St at Vine St,133,34.586466,32.330827
640,Woolsey St at Sacramento St,2042,42.017630,24.926543


The logic to assigning labels of station type is following:
- If the percent of rides in rush hour is more than 40 and less than 30, then I assign the label of "Commuter Hub"
- If the percent of rides on the weekend is more than or equal to 30% than it's a "Leisure Zone"
- If none of the above is true, then it become a "Hybrid Zone"

In [35]:
def classify_zone(row):
    if row["commute_pct"] > 40 and row["weekend_pct"] < 30:
        return "Commuter Hub"
    elif row["weekend_pct"] >= 30:
        return "Leisure Zone"
    else:
        return "Hybrid Zone"

In [36]:
station_profiles["station_type"] = station_profiles.apply(classify_zone, axis=1)

In [37]:
station_profiles.groupby("station_type").size()

station_type
Commuter Hub    354
Hybrid Zone     113
Leisure Zone    175
dtype: int64

In [38]:
master_df = pd.merge(station_profiles, df, on="start_station_name")

In [44]:
zone_analysis = master_df.groupby(["station_type", "rideable_type", "member_casual"]).agg(
    avg_rides_per_station=("total_rides", "mean"),
    med_rides_per_station=("total_rides", "median")
)

zone_analysis

avg_rides_per_station  \
station_type rideable_type member_casual                          
Commuter Hub classic_bike  casual                  15297.803409   
                           member                  16067.668945   
             electric_bike casual                  15593.685514   
                           member                  16268.914762   
Hybrid Zone  classic_bike  casual                   6938.069718   
                           member                   7604.955763   
             electric_bike casual                   7905.159634   
                           member                   8901.757426   
Leisure Zone classic_bike  casual                  13798.126770   
                           member                  15789.324479   
             electric_bike casual                  14942.645246   
                           member                  16847.485610   

                                          med_rides_per_station  
station_type rideable_type member_casual                         
Commuter Hub classic_bike  casual                       14430.0  
                           member                       15313.0  
             electric_bike casual                       14430.0  
                           member                       15313.0  
Hybrid Zone  classic_bike  casual                        4640.0  
                           member                        5974.0  
             electric_bike casual                        6632.0  
                           member                        7920.0  
Leisure Zone classic_bike  casual                       11178.0  
                           member                       14407.0  
             electric_bike casual                       12982.0  
                           member                       15730.0